# Multi-Source Illumination

An `NSQScene` can hold any number of sources simultaneously. All sources launch
rays in the same trace, and every ray carries flux proportional to its source's
`total_flux`. This makes it straightforward to model:

- **Multi-point LED arrays** — many PointSources at different positions
- **Off-axis illumination** — sources tilted relative to the optical axis
- **Field uniformity analysis** — sources spanning a range of field angles
- **Mixed spectral sources** — sources with different spectra

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from optiland.coordinate_system import CoordinateSystem
from optiland.nonsequential import (
    NSQScene, Spectrum,
    PointSourceConfig, CollimatedSourceConfig, ExtendedSourceConfig,
    IrradianceDetectorConfig,
    LensConfig,
)

spec_w = Spectrum(wavelengths=np.array([0.45, 0.55, 0.65]),
                  weights=np.array([1.0, 1.0, 1.0]))

## 1. LED Array — Multiple PointSources

Model a 3×3 LED array with 2 mm pitch. Each LED is a `PointSource` with a
narrow 30° emission half-angle, representing a Lambertian LED die.

In [2]:
pitch = 2.0  # mm between LEDs
scene_array = NSQScene()

# 3×3 grid of LEDs in the z=0 plane
for i, x in enumerate(np.linspace(-pitch, pitch, 3)):
    for j, y in enumerate(np.linspace(-pitch, pitch, 3)):
        scene_array.add_source(
            f'LED_{i}_{j}',
            CoordinateSystem(x=x, y=y, z=-20),
            PointSourceConfig(
                spectrum=spec_w, total_flux=1.0, half_angle_deg=30,
            ),
        )

# Single condenser lens to collimate the array
scene_array.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(r1=40, r2=-40, thickness=5, material='N-BK7', front_aperture_radius=15.0),
)

# Detector at the far field of the lens
scene_array.add_detector(
    'D', CoordinateSystem(z=80),
    IrradianceDetectorConfig(width=30, height=30, num_pixels_x=128, num_pixels_y=128),
)

result_array = scene_array.trace(num_rays=100_000, seed=42)
irr_array = result_array.detectors['D']

print(f"Total sources    : {len(scene_array.sources)}")
print(f"Total flux in    : {result_array.total_flux_in:.2f} W")
print(f"Flux detected    : {irr_array.total_flux:.4f} W")

fig = irr_array.plot(cmap='hot')
plt.title('3×3 LED array — irradiance at z=80 mm')
plt.tight_layout()
plt.show()
plt.close(fig)

Total sources    : 9
Total flux in    : 9.00 W
Flux detected    : 3.4789 W


C:\Users\kdani\AppData\Local\Temp\ipykernel_13564\2758724326.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Illumination Uniformity Analysis

A key figure of merit for illumination systems is **uniformity** — how evenly the
flux is distributed across the target. Common metrics:

- **Uniformity ratio** = min / max irradiance
- **RMS uniformity** = 1 − σ / μ
- **±X% area** = fraction of pixels within X% of the mean

In [3]:
E = irr_array.irradiance
# Only consider lit pixels (above 1% of peak)
lit = E > 0.01 * E.max()
E_lit = E[lit]

uniformity_min_max = E_lit.min() / E_lit.max()
rms_uniformity     = 1.0 - E_lit.std() / E_lit.mean()
within_20pct       = np.mean(np.abs(E_lit - E_lit.mean()) < 0.20 * E_lit.mean()) * 100

print(f"Uniformity (min/max)  : {uniformity_min_max:.3f}")
print(f"RMS uniformity (1-σ/μ): {rms_uniformity:.3f}")
print(f"Pixels within ±20% μ  : {within_20pct:.1f}%")

# Horizontal profile
ny, nx = E.shape
row_centre = E[ny // 2, :]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(irr_array.x_coords, row_centre)
ax.axhline(row_centre.mean(), color='r', linestyle='--', label='Mean (lit)')
ax.set_xlabel('x [mm]')
ax.set_ylabel('Irradiance [W/mm²]')
ax.set_title('Horizontal irradiance cross-section')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
plt.close(fig)

Uniformity (min/max)  : 0.010
RMS uniformity (1-σ/μ): 0.536
Pixels within ±20% μ  : 33.0%


C:\Users\kdani\AppData\Local\Temp\ipykernel_13564\1175378102.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Off-Axis and Tilted Sources

Tilt a source by setting `rx` and `ry` in the `CoordinateSystem` (values in
**radians**). This lets you model off-axis field angles or oblique illumination.

In [4]:
spec_green = Spectrum.monochromatic(0.55)
scene_tilt = NSQScene()

# On-axis source
scene_tilt.add_source(
    'S_on', CoordinateSystem(z=-80),
    CollimatedSourceConfig(spectrum=spec_green, total_flux=1.0, aperture_radius=8.0),
)
# 5-degree tilted source (field angle)
scene_tilt.add_source(
    'S_off', CoordinateSystem(z=-80, rx=np.radians(5)),
    CollimatedSourceConfig(spectrum=spec_green, total_flux=1.0, aperture_radius=8.0),
)

scene_tilt.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(r1=50, r2=-50, thickness=5, material='N-BK7', front_aperture_radius=15.0),
)
scene_tilt.add_detector(
    'D', CoordinateSystem(z=100),
    IrradianceDetectorConfig(width=20, height=20, num_pixels_x=128, num_pixels_y=128),
)

result_tilt = scene_tilt.trace(num_rays=60_000, seed=42)
irr_tilt = result_tilt.detectors['D']

fig = irr_tilt.plot(cmap='hot')
plt.title('On-axis + 5° off-axis collimated sources')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_13564\4000727631.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Mixed-Spectrum RGB Sources

Each source can have an independent `Spectrum`. This lets you simulate RGB
colour mixing or multi-wavelength systems.

In [5]:
spec_r = Spectrum.monochromatic(0.64)   # red
spec_g = Spectrum.monochromatic(0.532)  # green
spec_b = Spectrum.monochromatic(0.45)   # blue

scene_rgb = NSQScene()

# Three point sources at 3 mm separation, each a different colour
for name, spec, x in [('R', spec_r, -3.0), ('G', spec_g, 0.0), ('B', spec_b, 3.0)]:
    scene_rgb.add_source(
        name, CoordinateSystem(x=x, z=-50),
        PointSourceConfig(spectrum=spec, total_flux=1.0, half_angle_deg=20),
    )

scene_rgb.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(r1=50, r2=-50, thickness=5, material='N-BK7', front_aperture_radius=15.0),
)
scene_rgb.add_detector(
    'D', CoordinateSystem(z=90),
    IrradianceDetectorConfig(width=25, height=25, num_pixels_x=128, num_pixels_y=128),
)

result_rgb = scene_rgb.trace(num_rays=60_000, seed=42)
irr_rgb = result_rgb.detectors['D']

fig = irr_rgb.plot(cmap='hot')
plt.title('RGB point sources: mixing at detector')
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Total flux in    : {result_rgb.total_flux_in:.4f} W  (3 sources × 1 W each)")
print(f"Flux on detector : {irr_rgb.total_flux:.4f} W")

Total flux in    : 3.0000 W  (3 sources × 1 W each)
Flux on detector : 1.6714 W


C:\Users\kdani\AppData\Local\Temp\ipykernel_13564\952238651.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Extended Source for Diffuse Illumination

An `ExtendedSource` uniformly samples positions on a rectangle. Combined with a
condenser lens this approximates a Köhler illumination arrangement.

In [6]:
scene_ext = NSQScene()
scene_ext.add_source(
    'ES', CoordinateSystem(z=-50),
    ExtendedSourceConfig(
        spectrum=spec_green, total_flux=5.0,
        width=8, height=8, half_angle_deg=45,
    ),
)
scene_ext.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(r1=40, r2=-40, thickness=5, material='N-BK7', front_aperture_radius=15.0),
)
scene_ext.add_detector(
    'D', CoordinateSystem(z=80),
    IrradianceDetectorConfig(width=30, height=30, num_pixels_x=128, num_pixels_y=128),
)

result_ext = scene_ext.trace(num_rays=80_000, seed=0)
irr_ext = result_ext.detectors['D']

fig = irr_ext.plot(cmap='inferno')
plt.title(f'Extended source (8×8 mm) through condenser — {irr_ext.num_rays_hit:,} rays')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_13564\949386209.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

- Add as many sources as needed — all participate in the same trace
- Source position: `CoordinateSystem(x, y, z)` in mm
- Source tilt: `CoordinateSystem(rx=..., ry=...)` in radians
- Each source can have an independent `Spectrum` and `total_flux`
- `total_flux_in` in `SimulationResult` sums across all sources
- Analyse uniformity via `irradiance.min/max/std/mean` on the `IrradianceMap`